# Tutorial: VN Model Optimization and OOD Readiness

Prerequisites:
- Chay notebook khi thu muc hien tai dang o repo root, hoac `cd` vao repo truoc.

Learning goals:
- Xac dinh feature set anchor tot nhat.
- Kiem tra rank-sidecar co nen merge hay khong.
- Xem model VN hien tai da san sang dem sang JP, KR, US hay chua.


## Flow

1. Load report artifacts.
2. Show feature winner and key ablations.
3. Show rank-sidecar probes vs anchor.
4. Show OOD readiness on JP, KR, US.


In [1]:
from __future__ import annotations

from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


def find_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root. In Colab, cd vao repo roi chay lai.")


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def daily_rank_summary(run_name: str, model_name: str = "lstm_signmag_seed_42", split: str = "val") -> dict:
    pred_path = RUNS / run_name / "reports" / "core" / "predictions.csv"
    metrics_path = RUNS / run_name / "reports" / "core" / "metrics.json"
    history_path = RUNS / run_name / "history_signmag_seed_42.csv"
    pred_df = read_csv(pred_path)
    metrics = read_json(metrics_path)
    if pred_df.empty or model_name not in metrics:
        return {"run": run_name, "status": "missing"}

    pred_df = pred_df[(pred_df["model"] == model_name) & (pred_df["split"] == split)].copy()
    if pred_df.empty:
        return {"run": run_name, "status": "no_rows"}

    pred_df["Date"] = pd.to_datetime(pred_df["Date"])
    rows = []
    for date, group in pred_df.groupby("Date", sort=True):
        if len(group) < 5 or group["prediction"].nunique() < 2 or group["actual"].nunique() < 2:
            continue
        ic = group["prediction"].corr(group["actual"], method="spearman")
        ranks = group["prediction"].rank(method="first", pct=True)
        top = group.loc[ranks >= 0.75, "actual"]
        bottom = group.loc[ranks <= 0.25, "actual"]
        if len(top) == 0 or len(bottom) == 0:
            continue
        rows.append({"Date": date, "ic": ic, "quartile_return": float(top.mean() - bottom.mean())})

    daily = pd.DataFrame(rows)
    if daily.empty:
        return {"run": run_name, "status": "no_daily_metrics"}

    ic = daily["ic"].dropna()
    ret = daily["quartile_return"].dropna()
    ic_t = ic.mean() / (ic.std(ddof=1) / (len(ic) ** 0.5)) if len(ic) > 1 and ic.std(ddof=1) > 0 else None
    quartile_equity = (1.0 + ret).cumprod().iloc[-1] if not ret.empty else None
    epoch_count = len(read_csv(history_path)) if history_path.exists() else None
    return {
        "run": run_name,
        "epochs_logged": epoch_count,
        "val_rel_score": metrics[model_name][split]["rel_score"],
        "val_directional_accuracy": metrics[model_name][split]["directional_accuracy"],
        "mean_ic": float(ic.mean()),
        "ic_t_stat": float(ic_t) if ic_t is not None else None,
        "positive_ic_days": float((ic > 0).mean()),
        "quartile_equity": float(quartile_equity) if quartile_equity is not None else None,
    }


ROOT = find_root(Path.cwd())
RUNS = ROOT / "data" / "processed" / "assets" / "data_info_vn" / "history" / "training_runs"
REPORTS = RUNS / "reports"
ROOT


PosixPath('/Users/lap15111/Documents/research-paper/data_stock_market')

In [2]:
feature_main = read_csv(REPORTS / "feature_pruning" / "broad_signmag_prune_20260424_r04" / "summary.csv")
feature_ablation = read_csv(REPORTS / "feature_pruning" / "broad_signmag_prune_20260424_r05" / "summary.csv")
objective_smoke = read_csv(REPORTS / "feature_pruning" / "broad_signmag_prune_20260426_r02" / "summary.csv")

winner_cases = [
    "general_sector_full",
    "general_sector_breadth",
    "general_sector_momentum_relative",
    "general_sector_rank_return_alpha",
    "general_sector_rank_alpha",
    "general_sector_rank",
]
ablation_cases = [
    "general_sector_full",
    "general_sector_full_no_sector_rank",
    "general_sector_full_no_breadth",
    "general_sector_full_no_return_alpha",
    "general_sector_full_no_mom_gap",
    "general_sector_full_no_rank_pct",
]

print("Feature winner")
display(
    feature_main.loc[feature_main["case_name"].isin(winner_cases), ["case_name", "signmag_val_rel_score", "signmag_val_quartile_equity"]]
    .sort_values("signmag_val_rel_score", ascending=False)
    .reset_index(drop=True)
)

print("Key ablations from general_sector_full")
display(
    feature_ablation.loc[feature_ablation["case_name"].isin(ablation_cases), ["case_name", "signmag_val_rel_score", "signmag_val_quartile_equity"]]
    .sort_values("signmag_val_rel_score", ascending=False)
    .reset_index(drop=True)
)

print("Window and objective smoke")
display(
    objective_smoke[["case_name", "signmag_val_rel_score", "signmag_val_quartile_equity"]]
    .sort_values("signmag_val_rel_score", ascending=False)
    .reset_index(drop=True)
)


Feature winner


,case_name,signmag_val_rel_score,signmag_val_quartile_equity
0,general_sector_full,0.005340,2.477205
1,general_sector_breadth,0.004847,1.808564
2,general_sector_momentum_relative,0.002726,1.514155
3,general_sector_rank_alpha,0.002169,0.807625
4,general_sector_rank_return_alpha,0.002153,1.557663
5,general_sector_rank,0.001569,1.858921


Key ablations from general_sector_full


,case_name,signmag_val_rel_score,signmag_val_quartile_equity
0,general_sector_full,0.005340,2.477205
1,general_sector_full_no_sector_rank,0.003007,1.614769
2,general_sector_full_no_breadth,0.002594,1.157490
3,general_sector_full_no_rank_pct,0.002397,2.439854
4,general_sector_full_no_return_alpha,0.001158,1.676254
5,general_sector_full_no_mom_gap,-0.000415,1.433871


Window and objective smoke


,case_name,signmag_val_rel_score,signmag_val_quartile_equity
0,general_sector_full_obj_anchor_smoke,0.005340,2.477205
1,general_sector_full_w20_smoke,0.003225,1.355551
2,general_sector_full_weighted_smoke,0.002759,1.708018
3,general_sector_full_w10_smoke,0.001459,1.282528
4,general_sector_full_sharp_smoke,0.000379,2.636697


In [3]:
rank_df = pd.DataFrame(
    [
        daily_rank_summary("broad_signmag_prune_general_sector_full_20260424_r04"),
        daily_rank_summary("broad_signmag_general_sector_full_rank005_nogpu_20260428_r01"),
        daily_rank_summary("broad_signmag_general_sector_full_rank001_nogpu_20260428_r01"),
    ]
)

name_map = {
    "broad_signmag_prune_general_sector_full_20260424_r04": "anchor",
    "broad_signmag_general_sector_full_rank005_nogpu_20260428_r01": "rank_sidecar_0.05",
    "broad_signmag_general_sector_full_rank001_nogpu_20260428_r01": "rank_sidecar_0.01",
}
rank_df["case"] = rank_df["run"].map(name_map).fillna(rank_df["run"])

display(
    rank_df[[
        "case",
        "epochs_logged",
        "val_rel_score",
        "val_directional_accuracy",
        "mean_ic",
        "ic_t_stat",
        "positive_ic_days",
        "quartile_equity",
    ]]
    .sort_values("val_rel_score", ascending=False)
    .reset_index(drop=True)
)


,case,epochs_logged,val_rel_score,val_directional_accuracy,mean_ic,ic_t_stat,positive_ic_days,quartile_equity
0,anchor,12,0.005340,0.488539,0.051727,4.985974,0.585736,2.406571
1,rank_sidecar_0.05,9,0.001416,0.481536,0.036014,3.461938,0.575114,2.189831
2,rank_sidecar_0.01,14,-0.001105,0.487554,0.045545,4.533591,0.581184,1.489405


In [4]:
ood_reports = {
    "JP50": "broad_signmag_general_sector_full_r04__jp50_anchor_readiness",
    "KR50": "broad_signmag_general_sector_full_r04__kr50_anchor_readiness",
    "US100": "broad_signmag_general_sector_full_r04__us100_anchor_readiness",
}

rows = []
for market, report_name in ood_reports.items():
    payload = read_json(REPORTS / "ood_readiness" / report_name / "summary.json")
    if not payload:
        rows.append({"market": market, "status": "missing"})
        continue
    rows.append(
        {
            "market": market,
            "accepted_codes": payload.get("accepted_codes"),
            "test_rows": payload.get("test_rows"),
            "rel_score": payload.get("rel_score"),
            "directional_accuracy": payload.get("directional_accuracy"),
            "mean_spearman_ic": payload.get("mean_spearman_ic"),
            "ic_t_stat": payload.get("ic_t_stat"),
            "quartile_equity": payload.get("quartile_equity"),
            "quartile_hit_rate": payload.get("quartile_hit_rate"),
            "quartile_max_drawdown": payload.get("quartile_max_drawdown"),
        }
    )

display(pd.DataFrame(rows))


,market,accepted_codes,test_rows,rel_score,directional_accuracy,mean_spearman_ic,ic_t_stat,quartile_equity,quartile_hit_rate,quartile_max_drawdown
0,JP50,26,21366,-0.001755,0.499343,0.021809,2.446391,1.222140,0.533414,-0.179042
1,KR50,44,35968,-0.001095,0.488907,0.036369,5.097431,1.433111,0.539024,-0.274963
2,US100,89,75017,-0.001311,0.510576,0.005240,0.755295,1.110542,0.482800,-0.137430


## Quick Read

- `general_sector_full` van la feature set anchor tot nhat.
- Hai probe `rank sidecar` hien tai deu kem hon anchor, nen chua nen merge.
- KR giu duoc rank transfer tot nhat, JP trung binh, US yeu ro.
- Anchor hien tai van la `VN-first model`, chua san sang dem sang JP, KR, US de live.
